#1.Problem Statement

Identify the Electric Vehicle Demand, based on the available data.

#2.Exploratory Data Analysis

##2.1 Loading a dataset

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("/Volumes/electric_vehicle/ev_sales/ev_sales_data_raw/train.csv")

##2.2 Inspecting the dataset

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df["Daily_Commute_km"].describe()

###2.2.1 Checking for missing values

In [0]:
df.isna().sum()

###2.2.2 Slicing Data into groups

In [0]:
df.loc[0:10,["City_Type","Will_Buy_EV"]] #.loc screens by label

In [0]:
df.iloc[0:10, [9, 14]] #.iloc screens by index position

In [0]:
df.iloc[0:10, 9:14]

In [0]:
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

How many of these customers travel 30 km or more everyday?

In [0]:
thirty_km_commuters = df[df.Daily_Commute_km == 30]

In [0]:
thirty_km_commuters.shape

#### 2.2.2.1 Reshaping into conditional tables

In [0]:
Will_buy_EV_Dist = df["Will_Buy_EV"].value_counts(normalize=True)*100

In [0]:
Will_buy_EV_Dist

In [0]:
Annual_Income_EV_Yes = df.loc[df["Will_Buy_EV"] == "Yes","Annual_Income_USD"]

In [0]:
Annual_Income_EV_Yes.describe()

In [0]:
Annual_Income_EV_Yes = Annual_Income_EV_Yes.to_frame(name="Income")

In [0]:
Annual_Income_EV_Yes.head()

In [0]:
Annual_Income_EV_Yes = Annual_Income_EV_Yes.reset_index(drop=True)
Annual_Income_EV_Yes.head()

In [0]:
Annual_Income_EV_Yes = Annual_Income_EV_Yes.drop(columns="index", errors="ignore")

In [0]:
Annual_Income_EV_Yes.head()

In [0]:
Annual_income_EV_No=df.loc[df["Will_Buy_EV"] == "No","Annual_Income_USD"]
Annual_income_EV_No.describe()



In [0]:
Urban_income_group = df.loc[df["City_Type"] == "Urban",["Age","Annual_Income_USD","Daily_Commute_km","Will_Buy_EV"]]

In [0]:
Urban_income_group.shape

In [0]:

Urban_income_group["Will_Buy_EV"].value_counts(normalize=True)

In [0]:
df["City_Type"].unique()

**Create groups for suburban and rural customers.**

In [0]:
subrban_income_group = df.loc[df["City_Type"] == "Suburban",["Age","Annual_Income_USD","Will_Buy_EV"]]

In [0]:
rural_income_group = df.loc[df["City_Type"] == "Rural",["Age","Annual_Income_USD","Will_Buy_EV"]]

**Check the split of buyer preference in these groups**

In [0]:
Urban_income_group.value_counts("Will_Buy_EV")

In [0]:
Urban_income_group["Will_Buy_EV"].value_counts(normalize=True) * 100

In [0]:
Urban_income_group.describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "Yes"].describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "No"].describe()

**Observing Central Tendency across reshaped data**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].mean().round(2) #average

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].median().round(2) #middle salary point

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].std().round(2) #square root of the variance from the mean

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].quantile(.25).round(2)

**Creating Custom dataframes and renaming columns by summary statistics**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'count']).rename(columns={'mean': 'Avg_Annual_Income_USD', 'count': 'Count_People'})


In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'median', 'count', 'min', 'max']).rename(columns={'mean': 'Avg_Annual_Income_USD','median': 'Median_Annual_Income_USD','count': 'Count_People','min':'Min_Annual_Income_USD','max':'Max_Annual_Income_USD'})


In [0]:
# Split commuters into above and below 30 km
above_30 = df[df["Daily_Commute_km"] > 30]
below_30 = df[df["Daily_Commute_km"] <= 30]

above_30_pct = (above_30["Will_Buy_EV"].value_counts(normalize=True) * 100).round(2)
below_30_pct = (below_30["Will_Buy_EV"].value_counts(normalize=True) * 100).round(2)

commute_buy_preference = pd.DataFrame({
    "Commute > 30 km (%)": above_30_pct,
    "Commute <= 30 km (%)": below_30_pct,
})


commute_buy_preference

###2.2.3 Visualizing Distributions on unscaled data for business reporting 

In [0]:
df["y"] = df["Will_Buy_EV"].map({"Yes": 1, "No": 0}) #Convert to integer datatype

In [0]:
print(df["Will_Buy_EV"].unique())            # check the exact labels first



print(df["y"].isna().sum())                  # must be 0
print(df["y"].value_counts())

In [0]:
pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Select only numerical columns and compute the correlation matrix
numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

In [0]:

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

### 2.2.4 Transforming the data from qualitative to quantitative data types


In [0]:
df.info()

In [0]:
df["Gender"].value_counts() #inspecting the need for dummy variables

In [0]:
gender_dummies = pd.get_dummies(df["Gender"], prefix="Gender", dtype=int) #Creating the transformation of the gender column

In [0]:
new_cols = gender_dummies.loc[:, ~gender_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df = df.drop(["Gender_Female","Gender_Male","Gender_Other"], axis=1)

In [0]:
df[["Gender_Female", "Gender_Male", "Gender_Other", "y"]].corr()["y"]

In [0]:
df["City_Type"].value_counts()


In [0]:
city_type_dummies = pd.get_dummies(df["City_Type"], prefix="City_Type", dtype=int)



In [0]:
new_cols = city_type_dummies.loc[:, ~city_type_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df[["City_Type_Urban", "City_Type_Suburban", "City_Type_Rural", "y"]].corr()["y"]

In [0]:
df["Current_Car_Type"].value_counts()


In [0]:
current_car_type_dummies = pd.get_dummies(df["Current_Car_Type"], prefix="Current_Car_Type", dtype=int)

In [0]:
new_cols = current_car_type_dummies.loc[:, ~current_car_type_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df[["Current_Car_Type_Hatchback", "Current_Car_Type_SUV", "Current_Car_Type_Sedan", "Current_Car_Type_Truck","y"]].corr()["y"]

#### 2.2.4.1 Creating a function to automate transformation and parsing of the one-hot encoded/dummy variables to the dataframe

In [0]:
def add_dummies_to_df(df, columns):
    """Create dummy variables for the given columns and append them to df (skips columns already dummified, safe to re-run)."""
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' not found in df, skipping.")
            continue
        dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
        new_cols = dummies.loc[:, ~dummies.columns.isin(df.columns)]  # only dummy columns not already present
        if not new_cols.empty:
            df = pd.concat([df, new_cols], axis=1)
    return df



In [0]:
help(add_dummies_to_df) #reading the docstring we created

In [0]:
df = add_dummies_to_df(df, ["Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"])



In [0]:
df.head()

In [0]:
# Select only numerical columns and compute the correlation matrix
numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

In [0]:

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

In [0]:
import numpy as np

In [0]:
plt.figure(figsize=(14, 11))

# Mask the upper triangle so each pair is shown only once (less visual clutter)
# Drop the id column — it carries no meaningful correlation information
corr_plot = correlation_matrix.drop(index="id", columns="id", errors="ignore")

mask = np.triu(np.ones_like(corr_plot, dtype=bool))

sns.heatmap(
    corr_plot,
    mask=mask,
    annot=True,
    annot_kws={"size": 8},
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Correlation"},
)

plt.title("Correlation Heatmap (lower triangle)", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)

plt.tight_layout()

plt.show()

##2.3 Hypothesis Framing

## A. Main effects: what drives Will_Buy_EV

###2.3.1 H1: Buyers have higher environmental concern than non-buyers

In [0]:
# Box plot of Environmental Concern for buyers (y=1) vs non-buyers (y=0)
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df,
    x="y",
    y="Environmental_Concern_Level",
    hue="y",
    palette={0: "#1f77b4", 1: "#d62728"},
    legend=False,
)

plt.title("Environmental Concern: EV Buyers vs Non-Buyers", fontsize=14)
plt.xlabel("Bought EV")
plt.ylabel("Environmental Concern")
plt.xticks(ticks=[0, 1], labels=["Non-Buyers (0)", "Buyers (1)"])

plt.tight_layout()
plt.show()


In [0]:
# Histogram (100 bins) of Environmental Concern for buyers (y=1) vs non-buyers (y=0)
# to check the shape/normality of the distributions
plt.figure(figsize=(12, 6))

sns.histplot(
    data=df,
    x="Environmental_Concern_Level",
    hue="y",
    bins=30,
    element="step",       # outline style keeps overlapping bins readable
    stat="density",       # density makes the two groups comparable despite different sizes
    common_norm=False,    # normalize each group separately
    palette={0: "#1f77b4", 1: "#d62728"},
)

plt.title("Environmental Concern Distribution: EV Buyers vs Non-Buyers (100 bins)", fontsize=14)
plt.xlabel("Environmental Concern")
plt.ylabel("Density")
plt.legend(title="Bought EV", labels=["Non-Buyers (0)", "Buyers (1)"])

plt.tight_layout()
plt.show()


####2.3.1.1 Welch t-test for groups have inequal variances or inequal sample sizes

In [0]:
import numpy as np                                     
from scipy import stats

# Split Environmental Concern into the two groups: buyers (y=1) and non-buyers (y=0)
concern_buyers = df.loc[df["y"] == 1, "Environmental_Concern_Level"].dropna()
concern_non_buyers = df.loc[df["y"] == 0, "Environmental_Concern_Level"].dropna()

alpha = 0.05
conf_level = 1 - alpha                                 # 95% confidence

# Basic group statistics                                
n1, n2 = len(concern_buyers), len(concern_non_buyers)
m1, m2 = concern_buyers.mean(), concern_non_buyers.mean()
s1, s2 = concern_buyers.std(), concern_non_buyers.std()

# Welch's t-test (does NOT assume equal variances or equal sample sizes)
t_stat, p_value = stats.ttest_ind(concern_buyers, concern_non_buyers, equal_var=False)

# 95% CI for the DIFFERENCE in means (Welch method)
diff = m1 - m2
var_m1, var_m2 = s1**2 / n1, s2**2 / n2                # variance of each sample mean
se_diff = np.sqrt(var_m1 + var_m2)                     # standard error of the difference
dof = (var_m1 + var_m2)**2 / (var_m1**2 / (n1 - 1) + var_m2**2 / (n2 - 1))  # Welch–Satterthwaite df
t_crit = stats.t.ppf(1 - alpha / 2, dof)               # critical t value
ci_low, ci_high = diff - t_crit * se_diff, diff + t_crit * se_diff

# 95% CI for each group's own mean (per-group SE and t critical value)
se1, se2 = s1 / np.sqrt(n1), s2 / np.sqrt(n2)
tc1, tc2 = stats.t.ppf(1 - alpha / 2, n1 - 1), stats.t.ppf(1 - alpha / 2, n2 - 1)
b_low, b_high = m1 - tc1 * se1, m1 + tc1 * se1
nb_low, nb_high = m2 - tc2 * se2, m2 + tc2 * se2


# Output
print(f"Buyers (y=1):     n={n1:,}, mean={m1:.3f}, std={s1:.3f}, 95% CI [{b_low:.3f}, {b_high:.3f}]")
print(f"Non-buyers (y=0): n={n2:,}, mean={m2:.3f}, std={s2:.3f}, 95% CI [{nb_low:.3f}, {nb_high:.3f}]")
print(f"\nMean difference (buyers − non-buyers): {diff:.3f}")
print(f"95% CI for the difference:            [{ci_low:.3f}, {ci_high:.3f}]")
print(f"Relative difference:                  {diff / m2 * 100:.1f}% higher than non-buyers")
print(f"\nWelch's t-statistic: {t_stat:.4f}  (df = {dof:,.0f})")
print(f"p-value: {p_value:.2e}")                       # CHANGED: scientific notation

if p_value < alpha:
    print(f"\np < {alpha}: Significant difference in Environmental Concern between buyers and non-buyers.")
    print(f"The true difference is plausibly between {ci_low:.3f} and {ci_high:.3f} points.")
else:
    print(f"\np >= {alpha}: No significant difference in Environmental Concern between buyers and non-buyers.")
    print(f"The CI [{ci_low:.3f}, {ci_high:.3f}] includes 0, so 'no difference' cannot be ruled out.")


#### 2.3.1.2 Cohen's d Effect Size of Environmental Concern on Buyer behaviour

In [0]:

n1, n0 = len(concern_buyers), len(concern_non_buyers)
mean1, mean0 = concern_buyers.mean(), concern_non_buyers.mean()
var1, var0 = concern_buyers.var(ddof=1), concern_non_buyers.var(ddof=1)

# Cohen's d uses the pooled standard deviation (weighted by sample size)
pooled_sd = np.sqrt(((n1 - 1) * var1 + (n0 - 1) * var0) / (n1 + n0 - 2))
cohens_d = (mean1 - mean0) / pooled_sd

print(f"Buyers (y=1):     n={n1}, mean={mean1:.3f}, std={concern_buyers.std():.3f}")
print(f"Non-buyers (y=0): n={n0}, mean={mean0:.3f}, std={concern_non_buyers.std():.3f}")
print(f"Pooled SD: {pooled_sd:.3f}")
print(f"Cohen's d: {cohens_d:.3f}")

# Interpretation using Cohen's conventional thresholds
abs_d = abs(cohens_d)
if abs_d < 0.2:
    interpretation = "negligible"
elif abs_d < 0.5:
    interpretation = "small"
elif abs_d < 0.8:
    interpretation = "medium"
else:
    interpretation = "large"

print(f"Effect size interpretation: {interpretation} effect")

# Optional: 95% CI for Cohen's d via the standard error approximation (Hedges & Olkin)
se_d = np.sqrt((n1 + n0) / (n1 * n0) + cohens_d**2 / (2 * (n1 + n0)))
ci_low, ci_high = cohens_d - 1.96 * se_d, cohens_d + 1.96 * se_d
print(f"Approximate 95% CI for Cohen's d: [{ci_low:.3f}, {ci_high:.3f}]")


####2.3.1.3 Mann-Whitney U test for Robustness Check

In [0]:
from scipy import stats

# Mann-Whitney U test (non-parametric) as a robustness check for H1:
# does Environmental Concern still differ between buyers and non-buyers without assuming normality or equal variances?

# Medians for context (the Mann-Whitney U test compares distributions/ranks)
print(f"Buyers (y=1):     n={len(concern_buyers)}, median={concern_buyers.median():.3f}, mean rank summary")
print(f"Non-buyers (y=0): n={len(concern_non_buyers)}, median={concern_non_buyers.median():.3f}")

# Mann-Whitney U test (two-sided)
u_stat, p_value = stats.mannwhitneyu(concern_buyers, concern_non_buyers, alternative="two-sided")

# Rank-biserial correlation as the effect size for Mann-Whitney
rank_biserial = ((2 * u_stat)- 1)/ (n1 * n0)  # = (2U/(n1*n0)) - 1 with U for group 1
print(f"\nMann-Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.6f}")
print(f"Rank-biserial correlation (effect size): {rank_biserial:.3f}")

alpha = 0.05
if p_value < alpha:
    print(f"\np < {alpha}: The non-parametric test confirms a significant difference in Environmental "
          "Concern between buyers and non-buyers (robust to non-normality).")
else:
    print(f"\np >= {alpha}: The non-parametric test finds no significant difference in Environmental "
          "Concern between buyers and non-buyers.")


####2.3.1.4 DECISION ON H1: "ENVIRONMENTAL CONCERN HAS A SIGNIFICANT IMPACT ON BUYER BEHAVIOUR"

**Storyline for H1:** Environmental concern is the Single strong signal of EV purchase intent. Buyers score 1.75 points higher on environmental concern (4.38 vs 2.63), 66% higher than non-buyers. The estimate is highly precise (95% CI: 1.74–1.75).

BUSINESS IMPLICATIONS
- **Targeting:** 
Prioritise high-concern customers in campaigns. They are the most receptive audience.
- **Conversion:** for high-concern non-buyers, the problem is not the message. Find their barrier and remove it: subsidy information, home-charging installation, or range reassurance.
- **Messaging by segment:** lead with environmental benefits for high-concern customers. For low-concern customers, lead with cost savings and subsidies, which are your second-strongest driver by the correlation yet to be confirmed.

###2.3.2 H2: The Yes rate is higher when a subsidy is available

### 2.3.3 H3: The Yes rate differs across range-anxiety levels (Low > Medium > High)

### 2.3.4 H4: The Yes rate is higher when home charging is possible

### 2.3.5 H5: Buyers have higher annual income

### 2.3.6 H6: Buyers have shorter daily commutes

## B. “No effect” hypotheses: variables near zero

Gender has no meaningful effect on the Yes rate

Age has no meaningful effect on the Yes rate

City type has no meaningful direct effect on the Yes rate

##C. Relationships among the predictors

Urban residents are less likely to have home charging than rural or suburban residents

Urban residents have more public charging stations near home and work

People with home charging are more likely to report low range anxiety

People with longer commutes report higher range anxiety

## D. Interaction and mediation hypotheses

Subsidies raise the Yes rate more for lower-income people

Environmental concern turns into buying mainly when income is high enough

A long commute lowers the Yes rate less when home charging is possible

Home charging raises the Yes rate through lower range anxiety

Environmental concern, subsidy and income stay significant after controlling for all other variables